In [10]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\DELL\Practical Deep learning\Datasets\bluebook_bulldozer\TrainAndValid.csv")
df.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_4548\2608757952.py:1: DtypeWarning: Columns (13,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\DELL\Practical Deep learning\Datasets\bluebook_bulldozer\TrainAndValid.csv")


,SalesID,SalePrice,MachineID,ModelID,datasource,auctioneerID,YearMade,MachineHoursCurrentMeter,UsageBand,saledate,...,Undercarriage_Pad_Width,Stick_Length,Thumb,Pattern_Changer,Grouser_Type,Backhoe_Mounting,Blade_Type,Travel_Controls,Differential_Type,Steering_Controls
0,1139246,66000.0,999089,3157,121,3.0,2004,68.0,Low,11/16/2006 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139248,57000.0,117657,77,121,3.0,1996,4640.0,Low,3/26/2004 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
2,1139249,10000.0,434808,7009,121,3.0,2001,2838.0,High,2/26/2004 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1139251,38500.0,1026470,332,121,3.0,2001,3486.0,High,5/19/2011 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139253,11000.0,1057373,17311,121,3.0,2007,722.0,Medium,7/23/2009 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
thresh = 0.5 
df_filtered = df.loc[:, df.isnull().mean() <= thresh]
df_filtered.isna().sum()

SalesID                    0
SalePrice                  0
MachineID                  0
ModelID                    0
datasource                 0
auctioneerID           20136
YearMade                   0
saledate                   0
fiModelDesc                0
fiBaseModel                0
fiSecondaryDesc       140727
fiProductClassDesc         0
state                      0
ProductGroup               0
ProductGroupDesc           0
Enclosure                334
Hydraulics             82565
Coupler               192019
dtype: int64

In [4]:
clean_df = df_filtered.drop(columns = ['fiSecondaryDesc', 'Hydraulics', 'Coupler', 'fiModelDesc', 'fiBaseModel', 'fiProductClassDesc'])
clean_df.isna().sum()

SalesID                 0
SalePrice               0
MachineID               0
ModelID                 0
datasource              0
auctioneerID        20136
YearMade                0
saledate                0
state                   0
ProductGroup            0
ProductGroupDesc        0
Enclosure             334
dtype: int64

In [5]:
clean_df = clean_df.dropna()

In [6]:
def pre_process(df):
    NUM = ['SalesID', 'SalePrice', 'MachineID', 'ModelID', 'datasource',
       'auctioneerID', 'YearMade']
    CAT = ['state', 'ProductGroup', 'ProductGroupDesc',
       'Enclosure']
    df = pd.get_dummies(df, columns=CAT, drop_first=True)

    if 'saledate' in df.columns:
      df['saledate'] = pd.to_datetime(df['saledate'], errors='coerce')
      df['saleYear'] = df['saledate'].dt.year
      df['saleMonth'] = df['saledate'].dt.month
      df['saleDay'] = df['saledate'].dt.day
      df['saleDayOfWeek'] = df['saledate'].dt.dayofweek
      df = df.drop(columns=['saledate'])

    train_idx = clean_df['YearMade'] < 2000
    valid_idx = (clean_df['YearMade'] >= 2000) & (clean_df['YearMade'] < 2005)
    test_idx = clean_df['YearMade'] >= 2005

    train = df[train_idx]
    valid = df[valid_idx]
    test = df[test_idx]

    target_cols = 'SalePrice'

    x_train = train.drop(columns=[target_cols])
    y_train = train[target_cols]
    x_valid = valid.drop(columns=[target_cols])
    y_valid = valid[target_cols]
    x_test = test.drop(columns=[target_cols])
    y_test = test[target_cols]


    return x_train, y_train, x_valid, y_valid, x_test, y_test   

In [7]:
x_train, y_train, x_valid, y_valid, x_test, y_test = pre_process(clean_df)

In [14]:
model_t = DecisionTreeRegressor()
tree = model_t.fit(x_train, y_train)

In [21]:
def get_metrics(y_valid, model):
    preds = model.predict(x_valid)
    mse = mean_squared_error(y_valid, preds)
    r2 = r2_score(y_valid, preds)
    print(f''' 
Mean Squared Error for {model} : {mse}
R2_score for {model} : {r2}''')

In [22]:
model_f = RandomForestRegressor(n_estimators=100, random_state=42)
forest = model_f.fit(x_train, y_train)

In [23]:
forest_metrics = get_metrics(y_valid, forest)
tree_metrics = get_metrics(y_valid, tree)

 
Mean Squared Error for RandomForestRegressor(random_state=42) : 238020782.9130524
R2_score for RandomForestRegressor(random_state=42) : 0.6278748853113947
 
Mean Squared Error for DecisionTreeRegressor() : 312786915.24378943
R2_score for DecisionTreeRegressor() : 0.5109844389062912
